# 02 · Monte Carlo: portföyün olası gelecekleri

Tek bir tahmin ("THYAO 300 olur") yerine **binlerce olası yol** üretip sonuçların dağılımına bakarız:
"1 ay sonra portföyün %5 ihtimalle X TL'den fazla kaybeder" (VaR) gibi.

**Öğreneceklerin:** geometrik Brown hareketi (GBM), Itô düzeltmesi, korelasyonlu simülasyon (Cholesky),
tarihsel bootstrap, VaR ve CVaR (Expected Shortfall), VaR geriye dönük testi.

Önce 01'i bitir (`data/prices_adj.csv` lazım).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))  # learn/ klasöründen çalıştır
import kontrol as K
import papidata as pdx

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (11, 4)

In [ ]:
px = pd.read_csv(pdx.DATA_DIR / "prices_adj.csv", index_col=0, parse_dates=True)
pos = pdx.positions()
TICK = list(pos.index)  # portföydeki semboller
r = np.log(px[TICK]).diff()
r3 = r["2023-11-01":].dropna()  # son 3 yıl, hepsinin verisi olan günler
pos

## 1. GBM: tek hisse

Model: dS = μS dt + σS dW. Çözümü:
S_{t+Δt} = S_t · exp((μ − σ²/2)Δt + σ√Δt · Z), Z ~ N(0, 1).

**Itô düzeltmesi (−σ²/2):** fiyatın beklenen değeri e^(μt) ile büyüsün diye log getirinin ortalaması
μ değil μ − σ²/2 olmalı. Oynak bir hissede "ortalama yol" ile "ortanca yol" bu yüzden ayrışır.
Δt = 1/252 (bir işlem günü).

In [ ]:
def gbm_paths(s0: float, mu: float, sigma: float, days: int, n_paths: int, rng: np.random.Generator) -> np.ndarray:
    """GBM fiyat yolları; şekil (n_paths, days + 1), ilk sütun s0.

    mu, sigma YILLIK (ör. 0.30, 0.40). Döngü yazma: tüm Z'leri tek seferde üret, np.cumsum kullan.
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.gbm_paths(gbm_paths)

In [ ]:
rng = np.random.default_rng(42)
mu = r3["THYAO"].mean() * 252 + r3["THYAO"].var() * 252 / 2  # log ortalamadan μ'ye
sigma = r3["THYAO"].std() * np.sqrt(252)
s0 = px["THYAO"].dropna().iloc[-1]
paths = gbm_paths(s0, mu, sigma, 252, 10_000, rng)
print(f"μ = %{mu*100:.1f}, σ = %{sigma*100:.1f}, S0 = {s0:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(paths[:60].T, lw=0.6); a1.set_title("THYAO: 60 olası yıl")
a2.hist(paths[:, -1], bins=100); a2.axvline(pos.loc["THYAO", "avg_cost"], color="r", label="maliyetin")
a2.set_title("1 yıl sonraki fiyat dağılımı"); a2.legend()
print(f"1 yıl sonra maliyetinin ({pos.loc['THYAO', 'avg_cost']}) altında kalma olasılığı: "
      f"%{(paths[:, -1] < pos.loc['THYAO', 'avg_cost']).mean()*100:.1f}")

**Dikkat:** μ'yi geçmişten tahmin etmek çok gürültülüdür (3 yılın ortalaması ±%20 hata payı taşır),
ve TL'de enflasyon μ'yi şişirir. Monte Carlo'nun asıl değerli çıktısı **dağılımın genişliği** (risk);
ortalaması değil.

## 2. Portföy: korelasyonlu simülasyon

Hisseler birlikte hareket eder. Bağımsız simüle edersek riski olduğundan **düşük** gösteririz.
Çözüm: Σ = L Lᵀ (Cholesky). Bağımsız Z'leri L ile çarparsan X = μ + Z Lᵀ, kovaryansı Σ olan getiriler olur.

In [ ]:
def correlated_normals(mean: np.ndarray, cov: np.ndarray, n_days: int, n_paths: int, rng: np.random.Generator) -> np.ndarray:
    """Günlük çok değişkenli normal getiriler; şekil (n_paths, n_days, n_assets).

    np.linalg.cholesky kullan (np.random.multivariate_normal YOK — mekanizmayı görmek için).
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.correlated_normals(correlated_normals)

## 3. Tarihsel bootstrap: kalın kuyruklar

Normal dağılım büyük düşüşleri az tahmin eder (30 Ekim'deki −%3,6 normalde "nadir" görünür, BIST'te
değil). Alternatif: **geçmiş günleri rastgele seçip** yeniden dizmek. Bir günün TÜM varlık getirilerini
birlikte seçersek korelasyon da kendiliğinden korunur.

In [ ]:
def bootstrap_rows(hist: pd.DataFrame, n_days: int, n_paths: int, rng: np.random.Generator) -> np.ndarray:
    """Geçmiş günlük getiri satırlarından (iadeli) örnekle; şekil (n_paths, n_days, n_assets)."""
    # TODO
    raise NotImplementedError

In [ ]:
K.bootstrap_rows(bootstrap_rows, r3)

In [ ]:
def portfolio_paths(sim: np.ndarray, last_prices: pd.Series, qty: pd.Series) -> np.ndarray:
    """Simüle log getirilerden (n_paths, n_days, n_assets) portföy değeri yolları (n_paths, n_days + 1)."""
    cum = np.concatenate([np.zeros(sim.shape[:1] + (1,) + sim.shape[2:]), np.cumsum(sim, axis=1)], axis=1)
    return (np.exp(cum) * (last_prices * qty).to_numpy()).sum(axis=2)

H = 21  # 1 ay ≈ 21 işlem günü
rng = np.random.default_rng(7)
last = px[TICK].ffill().iloc[-1]
qty = pos["quantity"]
v_norm = portfolio_paths(correlated_normals(r3.mean().to_numpy(), r3.cov().to_numpy(), H, 20_000, rng), last, qty)
v_boot = portfolio_paths(bootstrap_rows(r3, H, 20_000, rng), last, qty)
v0 = v_norm[0, 0]
pnl_norm, pnl_boot = v_norm[:, -1] - v0, v_boot[:, -1] - v0
print(f"Bugünkü portföy değeri: {v0:,.0f} TL")
plt.hist([pnl_norm, pnl_boot], bins=120, label=["normal", "bootstrap"], histtype="step")
plt.axvline(0, color="k", lw=0.5); plt.legend(); plt.title("1 aylık kâr/zarar dağılımı (TL)");

## 4. VaR ve CVaR

- **VaR(95%)**: kayıpların %95'inden büyük, %5'inden küçük olan eşik. "20 aydan 1'inde en az bu kadar kaybederim."
- **CVaR / Expected Shortfall(95%)**: o en kötü %5'lik dilimdeki **ortalama** kayıp. Kuyruğun derinliğini
  söyler; bankacılık düzenlemeleri (Basel) bu yüzden ES'ye geçti.

İkisini de **pozitif kayıp** olarak döndür (ör. VaR = 25.000 TL).

In [ ]:
def var_cvar(pnl: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    """(VaR, CVaR) — pozitif kayıp tutarları. pnl: simüle kâr/zarar (negatif = zarar)."""
    # TODO
    raise NotImplementedError

In [ ]:
K.var_cvar(var_cvar)
tbl = pd.DataFrame({
    name: {f"VaR {int(l*100)}": var_cvar(p, l)[0] for l in (0.95, 0.99)} | {f"CVaR {int(l*100)}": var_cvar(p, l)[1] for l in (0.95, 0.99)}
    for name, p in (("normal", pnl_norm), ("bootstrap", pnl_boot))
})
(tbl / v0 * 100).round(2).astype(str) + " %"  # portföyün yüzdesi olarak

## 5. Model doğru mu? VaR geriye dönük testi

Bir VaR modeli "%5 ihtimalle aşılır" diyorsa, geçmişte günlerin ~%5'inde aşılmış olmalı. Çok fazlaysa
riski küçümsüyor, çok azsa gereksiz temkinli. **Kritik kural:** t günü için VaR'ı sadece t'den ÖNCEKİ
verilerle hesapla (yoksa geleceği kullanmış olursun: look-ahead).

In [ ]:
def var_backtest(r: pd.Series, window: int = 250, level: float = 0.95) -> pd.DataFrame:
    """Günlük tarihsel VaR testi.

    Her gün için: var = önceki `window` günün getirilerinden VaR (pozitif kayıp),
    loss = o günün kaybı (−getiri), breach = loss > var (VaR yoksa NaN).
    Döndür: index = tarih, sütunlar var, loss, breach.
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.var_backtest(var_backtest, r["THYAO"].dropna())
bt = var_backtest(np.log(px["XU100"]).diff().dropna())
ax = (-bt["loss"]).plot(lw=0.5, label="günlük getiri")
(-bt["var"]).plot(ax=ax, label="−VaR 95")
ax.scatter(bt.index[bt["breach"] == 1], -bt["loss"][bt["breach"] == 1], s=6, c="r", label="aşım")
ax.legend(); ax.set_title(f"XU100 VaR testi — aşım oranı %{bt['breach'].mean()*100:.1f}");

## 6. Yorumla

1. Normal ve bootstrap VaR/CVaR arasındaki fark nereden geliyor? Hangisine daha çok güvenirsin, neden?
2. TI1 portföyün %55'i. Onu çıkarıp sadece hisselerle simüle edersen VaR yüzdesi ne olur? (Kodu değiştir, dene.)
3. VaR aşımları zamanda kümeleniyor mu (2020 Mart, 2026 Eylül fon krizi)? Bu, "günler bağımsız"
   varsayımı hakkında ne söylüyor? (İleri konu: GARCH.)
4. Bir yatırımcıya "1 ayda %95 VaR'ın X TL" demek ile "CVaR'ın Y TL" demek arasındaki farkı tek cümleyle anlat.

*Cevaplar:*

## papi'ye taşıma (sonra)

PRD M3: "forecasts + outcome tracker". Buradaki simülasyon, `papi sim montecarlo` gibi bir komut ve
dashboard'da portföy VaR kartı olabilir; tahminler **dondurulup** sonra gerçekleşenle karşılaştırılır
(papi'de `forecast` / `outcome` tabloları bunun için var).